# Manual RTL Design + Explanation (ChipChat Example 2: Sequence Detector)

This notebook provides a **hand-written** RTL implementation for the ChipChat sequence detector example and verifies it using the official testbench.

Evidence included:
- Commented testbench code and invocation method
- Manual RTL (hand-written)
- Exact `iverilog/vvp` commands and passing output


## Install iverilog

In [7]:
!apt-get -qq update
!apt-get -qq install -y iverilog

!iverilog -V | head -n 2
print(" iverilog installed and ready.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Icarus Verilog version 11.0 (stable) ()

Unable to get version from "/usr/lib/x86_64-linux-gnu/ivl/ivl -V -C"/tmp/ivrlh57e9c739" -C"/usr/lib/x86_64-linux-gnu/ivl/vvp.conf""
 iverilog installed and ready.


In [8]:
import os, textwrap, subprocess

ROOT = "/content/manual_partIb_example2"
os.makedirs(ROOT, exist_ok=True)
os.chdir(ROOT)

print(" Working directory:", os.getcwd())
!ls -la

 Working directory: /content/manual_partIb_example2
total 8
drwxr-xr-x 2 root root 4096 Feb 15 17:50 .
drwxr-xr-x 1 root root 4096 Feb 15 17:50 ..


## Testbench (commented) + How it is invoked

We use the ChipChat Example 2 testbench and prepend a short comment header for clarity.

Invocation used:
```bash
iverilog -g2012 -o manual_sim.out manual_sequence_detector.sv sequence_detector_tb_commented.v
vvp manual_sim.out


## Download testbench + create commented version

In [9]:
TESTBENCH_URL = "https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v"

# Download original TB
!curl -L -o sequence_detector_tb.v "$TESTBENCH_URL"

with open("sequence_detector_tb.v", "r") as f:
    orig_tb = f.read()

comment_header = """`timescale 1ns/1ps
// NOTE (Part I(b) - Manual RTL):
// This file is the ChipChat Example 2 testbench for the sequence detector.
// The original testbench is included unchanged below.
//
// How to run (exact commands):
//   iverilog -g2012 -o manual_sim.out manual_sequence_detector.sv sequence_detector_tb_commented.v
//   vvp manual_sim.out
//
// Expected DUT interface:
//   module sequence_detector(
//       input  wire       clk,
//       input  wire       reset_n,
//       input  wire [2:0] data,
//       output reg        sequence_found
//   );
"""

with open("sequence_detector_tb_commented.v", "w") as f:
    f.write(comment_header + "\n" + orig_tb)

print("✅ Testbench downloaded from:")
print(TESTBENCH_URL)

print("\n sequence_detector_tb_commented.v ")
with open("sequence_detector_tb_commented.v", "r") as f:
    print(f.read())

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2114  100  2114    0     0   8003      0 --:--:-- --:--:-- --:--:--  8007
✅ Testbench downloaded from:
https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v

 sequence_detector_tb_commented.v 
`timescale 1ns/1ps
// NOTE (Part I(b) - Manual RTL):
// This file is the ChipChat Example 2 testbench for the sequence detector.
// The original testbench is included unchanged below.
//
// How to run (exact commands):
//   iverilog -g2012 -o manual_sim.out manual_sequence_detector.sv sequence_detector_tb_commented.v
//   vvp manual_sim.out
//
// Expected DUT interface:
//   module sequence_detector(
//       input  wire       clk,
//       input  wire       reset_n,
//       input  wire [2:0] data,
//       output reg        sequence_foun

## Manual RTL

Design choices:
- Use a 3-stage shift register to store prior input samples.
- Assert `sequence_found` as a 1-cycle pulse when the previous samples match:
  `000, 110, 110` (in order).
- Evaluate detection **before** shifting in the current cycle’s input for correct timing.
- Use active-low asynchronous reset (`reset_n`) to match the testbench behavior.


In [10]:
%%writefile manual_sequence_detector.sv
`timescale 1ns/1ps

// Part I(b): Manual RTL design (hand-written)
// Strategy: 3-sample history window using shift registers.
// sequence_found pulses when the previous samples are 000, 110, 110 (in order).
// Important: detection is evaluated BEFORE shifting in the current cycle's data.

module sequence_detector(
    input  wire       clk,
    input  wire       reset_n,          // active-low reset (matches TB)
    input  wire [2:0] data,
    output reg        sequence_found
);

    // d1 = previous cycle sample, d2 = 2 cycles ago, d3 = 3 cycles ago
    reg [2:0] d1, d2, d3;

    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            d1 <= 3'b000;
            d2 <= 3'b000;
            d3 <= 3'b000;
            sequence_found <= 1'b0;
        end else begin
            // Detect based on prior samples (timing aligned to TB checks)
            sequence_found <= (d3 == 3'b000) &&
                              (d2 == 3'b110) &&
                              (d1 == 3'b110);

            // Shift in the new sample after detection
            d3 <= d2;
            d2 <= d1;
            d1 <= data;
        end
    end

endmodule


Writing manual_sequence_detector.sv


## Compile + simulate (prints exact commands + passing output)

In [11]:
tb_file  = "sequence_detector_tb_commented.v"
dut_file = "manual_sequence_detector.sv"
exe_file = "manual_sim.out"

compile_cmd = f"iverilog -g2012 -o {exe_file} {dut_file} {tb_file}"
sim_cmd     = f"vvp {exe_file}"

print(" Part I(b): EXACT COMMANDS ")
print(compile_cmd)
print(sim_cmd)

comp = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
print("\ncompile stdout\n", comp.stdout)
print("\ncompile stderr\n", comp.stderr)
assert comp.returncode == 0, "Compile failed. Check errors above."

sim = subprocess.run(sim_cmd, shell=True, capture_output=True, text=True)
print("\n stdout (PASSING OUTPUT) \n", sim.stdout)
print("\n sim stderr\n", sim.stderr)

assert sim.returncode == 0 and ("All test cases passed" in sim.stdout), "Simulation did not PASS."
print("\n✅ Part I(b) manual RTL PASSES the testbench.")


 Part I(b): EXACT COMMANDS 
iverilog -g2012 -o manual_sim.out manual_sequence_detector.sv sequence_detector_tb_commented.v
vvp manual_sim.out

compile stdout
 

compile stderr
 

 stdout (PASSING OUTPUT) 
 All test cases passed.


 sim stderr
 

✅ Part I(b) manual RTL PASSES the testbench.


## Explanation of design choices

I implemented the sequence detector using a **three-stage shift register** (`d1`, `d2`, `d3`) to store the previous three sampled input values. This approach avoids explicit FSM state encoding and aligns directly with the testbench’s cycle-based checking behavior. The detection logic is evaluated **before shifting in the current input** to ensure the output pulse occurs at the exact cycle expected by the testbench. I used an **active-low asynchronous reset** (`negedge reset_n`) to match the reset behavior driven by the provided testbench. The design is fully synthesizable and uses only sequential logic with non-blocking assignments.
